# **Hands-on Lab: Interactive Visual Analytics with Folium**

Estimated time needed: **40** minutes

The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.

In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.

## Objectives

This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:

In [1]:
import folium
import pandas as pd

In [2]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [3]:
URL = 'C:/Users/MARCELA JAIMES/Documents/D/DataScienceIBM/spacex_launch_geo.csv'
spacex_df= pd.read_csv(URL)

In [4]:
Min_payload = spacex_df['Payload Mass (kg)'].min()
Max_payload = spacex_df['Payload Mass (kg)'].max()
print(f'la carga mínima es {Min_payload} y la carga máxima es {Max_payload}')

la carga mínima es 0.0 y la carga máxima es 9600.0


Now, you can take a look at what are the coordinates for each site.

In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
Launch_Sites = launch_sites_df["Launch Site"]
Launch_Sites

0     CCAFS LC-40
1    CCAFS SLC-40
2      KSC LC-39A
3     VAFB SLC-4E
Name: Launch Site, dtype: object

In [7]:
spacex_df.shape

(56, 4)

Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.

In [8]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,

In [24]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.

Now, let's add a circle for each launch site in data frame `launch_sites`

An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`

An example of folium.Marker:

`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`

In [10]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


The generated map with marked launch sites should look similar to the following:

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>

Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.

# Task 2: Mark the success/failed launches for each site on the map

In [11]:

marker_cluster = MarkerCluster().add_to(site_map)

for _, row in spacex_df.iterrows():
     circle_marker=folium.CircleMarker(
        location=[row['Lat'], row['Long']],
        radius=5,
        color='green' if row['class'] == 1 else 'red',
        fill=True,
        fill_color='green' if row['class'] == 1 else 'red',
        fill_opacity=0.7
     )
     marker_cluster.add_child(circle_marker)
    
site_map
        

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates. Recall that data frame spacex_df has detailed launch records, and the class column indicates if this launch was successful or not

In [12]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    marker = folium.Marker(location=[record['Lat'],record['Long']],
    # Create an icon as a text label
       icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html = f'<div style="font-size: 12px; color:#d35400;"><b>{record["Launch Site"]}</b></div>',
        )
    )
    marker_cluster.add_child(marker)

site_map

Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`

Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object

'marker_cluster = MarkerCluster().add_to(site_map)'

# TASK 3: Calculate the distances between a launch site to its proximities

Next, we need to explore and analyze the proximities of launch sites.

Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)

In [13]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.

In [14]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.

In [15]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

In [16]:
distance = calculate_distance(34.4178, -119.72628, 34.6292, -120.61725)
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 


mid_lat = (34.4178 + 34.6292) / 2
mid_lon = (-119.72628 + -120.61725) / 2


In [17]:
coordinate = [mid_lat, mid_lon]

from folium.features import DivIcon

distance_marker = folium.Marker(
    location=coordinate,
    icon=DivIcon(
        icon_size=(150, 36),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance:.2f} KM</b></div>'
    )
)

In [23]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate

coordinates = [
    (34.4178, -119.72628),  # Punto A is Santa Barbara, which is a Coastal City
    (34.6292, -120.61725)   # Punto B is Vandenberg Space Force Base
]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)
site_map.add_child(distance_marker)

In [19]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site

In [20]:
distance = calculate_distance(34.42613, -119.84679, 34.6292, -120.61725)
# Create and add a folium. Marker on your selected closest coastline point on the map
# Display the distance between the coastline point and launch site using the icon property 
# Hear mid_lat and mid_long takes the coordinates regarding Goleta

mid_lat = 34.42613
mid_lon = -119.84679


In [21]:
coordinate = [mid_lat, mid_lon]

from folium.features import DivIcon

distance_marker = folium.Marker(
    location=coordinate,
    icon=DivIcon(
        icon_size=(150, 36),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance:.2f} KM</b></div>'
    )
)

In [22]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate

coordinates = [
    (34.42613, -119.84679),  # Punto A is Goleta, which is a City
    (34.6292, -120.61725)   # Punto B is Vandenberg Space Force Base
]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)
site_map.add_child(distance_marker)

In [ ]:
After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
No, none of them.

*   Are launch sites in close proximity to highways?
No, none of them

*   Are launch sites in close proximity to coastline?
Yes, all of them. 

*   Do launch sites keep a certain distance away from cities? 
No, none of them.


Also please try to explain your findings: I analyzed the distance between Vandenberg Space Force Base and Goleta. 
And I calculated the distance between them, and it is 74.13 km, and it is not critical. Furthermore, I analyzed the distance between the same 
Space Force Base and Santa Barbara city, and the distance is longer than the one between the same Space Force Base and Goleta, so neither does it.
